In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from harbor.analysis.cross_docking import DockingDataModel
from plotting_params import *

## input files

In [ ]:
posit_raw = DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_combined_results.parquet")

In [ ]:
posit_results = Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/all_evals_posit_combined_results.csv")

In [ ]:
pdf = pd.read_csv(posit_results)
pdf["Error_Lower"] = pdf["Fraction"] - pdf["CI_Lower"]
pdf["Error_Lower"] = pdf["Error_Lower"].apply(lambda x: 0 if x < 0 else x)
pdf["Error_Upper"] = pdf["CI_Upper"] - pdf["Fraction"]
pdf["Error_Upper"] = pdf["Error_Upper"].apply(lambda x: 0 if x < 0 else x)

In [ ]:
fdf = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/all_evals_fred_combined_results.csv")

## output files

In [ ]:
figpath = Path("../figures")
figpath.mkdir(exist_ok=True)

# Plot Similarity Split

In [ ]:
pdfsim = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/analyzed_results/all_evals_posit_combined_results.csv")
simdf = pdfsim[pdfsim["PairwiseSplit"] == "SimilaritySplit"]
fdfsim = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/analyzed_results/all_evals_fred_combined_results.csv")
fsimdf = fdfsim[fdfsim["PairwiseSplit"] == "SimilaritySplit"]
fsimdf["Method"] = "FRED"
simdf["Method"] = "POSIT"
simdf = pd.concat([simdf, fsimdf])

In [ ]:
simdf.nunique()

In [ ]:
sns.lineplot(simdf, x="Similarity_Threshold", y="Fraction", hue="Method", style="Score")

In [ ]:
simmax = simdf.groupby(["Similarity_Threshold", "Method", "Score"]).max().reset_index()

In [ ]:
sns.lineplot(simmax, x="Similarity_Threshold", y="Fraction", hue="Method", style="Score")

In [ ]:
plt = plot_filled_in_error_bars( raw_df=simmax, style_var="Score", color_var="Method", x_var="Similarity_Threshold",)
label_map.update({"Method": "Docking Algorithm", "POSIT": "OpenEye POSIT Docker", "FRED": "OpenEye FRED Docker"})
plt = update_labels(plt, label_map, x_label="TanimotoCombo Similarity", legend_subtitles=["Method", "Score"])
save_figure(plt, figpath / "similarity")